# 05 · Poetry: The Complete Guide

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 100 min

> ❗ Not in either source. **Poetry** is a mature, widely used tool for dependency management **and** packaging, in many company codebases. This notebook was executed with **Poetry 2.5**; differences from the popular Poetry 1.x style are marked ⚠️.

### 🎯 What you will learn (every major feature)
1. Installing Poetry correctly (and why not with plain `pip`)
2. `poetry new` / `poetry init`, project layout, the **Poetry 2 `pyproject.toml`** format
3. Version constraints: caret `^`, tilde `~`, wildcards, and how Poetry 2 writes them
4. `add` / `remove` with groups, extras, markers, Git/path/URL sources
5. The lock file: `lock`, `install`, `sync`, `update`, `check`
6. Virtual environments: `env info/list/use/remove/activate`, `run`, in-project venvs
7. Inspecting: `show` (`--tree`, `--latest`, `--outdated`, `--why`), `search`, `debug`
8. Scripts, `version` bumping, `build`, `publish`, private **sources** & credentials
9. Configuration (`poetry config`), plugins (`poetry self`), `export`, `shell`
10. Poetry in Docker/CI, Poetry vs uv, and migration

> 🧪 **Sandbox:** the next cell points Poetry's cache, config and data directories into a temporary folder and creates virtualenvs inside each project (`.venv`), so nothing touches your real Poetry setup.

In [1]:
import os, sys, tempfile, pathlib, shutil
LAB = pathlib.Path(tempfile.mkdtemp(prefix="poetry-lab-"))
os.environ.update({
    "LAB": str(LAB),
    "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"],
    "POETRY_CACHE_DIR": str(LAB / ".cache"),
    "POETRY_CONFIG_DIR": str(LAB / ".config"),
    "POETRY_DATA_DIR": str(LAB / ".data"),
    "POETRY_VIRTUALENVS_IN_PROJECT": "true",   # create .venv inside the project (like uv)
    "NO_COLOR": "1", "GIT_CONFIG_GLOBAL": os.devnull,
})
for var in ("FORCE_COLOR", "CLICOLOR_FORCE"):
    os.environ.pop(var, None)
print("poetry found at:", shutil.which("poetry"))

poetry found at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/poetry


In [2]:
%%bash
poetry --version

Poetry (version 2.5.1)


## 1. Installing Poetry

Poetry must live in **its own isolated environment**, never in your project's venv:
```bash
pipx install poetry                                   # ✅ recommended
uv tool install poetry                                # ✅ also great
curl -sSL https://install.python-poetry.org | python3 -   # official installer
```
Upgrade with `pipx upgrade poetry` / `poetry self update`. Shell completions: `poetry completions zsh > ~/.zfunc/_poetry`.

> ⚠️ `pip install poetry` into a project venv mixes Poetry's dependencies with your project's: avoid it.

## 2. Creating a project

| Command | Purpose |
|---|---|
| `poetry new NAME` | new project folder with `src/` layout, `tests/`, README, `pyproject.toml` |
| `poetry new --flat NAME` | flat layout instead |
| `poetry init` | add a `pyproject.toml` to an **existing** folder (interactive; `-n` for defaults) |

In [3]:
%%bash
cd "$LAB"
poetry new --author "Asha Rao <asha@example.com>" --python ">=3.12" bookshelf
cd bookshelf
find . -type f | sort
echo "--------- pyproject.toml"
cat pyproject.toml

Created package bookshelf in bookshelf


./README.md
./pyproject.toml
./src/bookshelf/__init__.py
./tests/__init__.py


--------- pyproject.toml


[project]
name = "bookshelf"
version = "0.1.0"
description = ""
authors = [
    {name = "Asha Rao",e

mail = "asha@example.com"}
]
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
]

[to

ol.poetry]
packages = [{include = "bookshelf", from = "src"}]

[build-system]
requires = ["poetry-co

re>=2.0.0,<3.0.0"]
build-backend = "poetry.core.masonry.api"


### The Poetry 2 `pyproject.toml`

Poetry 2 (2025+) uses the **standard `[project]` table** (PEP 621), so other tools (uv, pip, build) understand your metadata. Poetry-specific settings live under `[tool.poetry]`:

```toml
[tool.poetry]
packages = [{ include = "bookshelf", from = "src" }]   # what goes into the wheel
# also: include/exclude files, scripts with extras, requires-poetry, ...
```

> ⚠️ **Poetry 1.x format (legacy, still common in existing repos):**
> ```toml
> [tool.poetry]
> name = "bookshelf"
> version = "0.1.0"
> authors = ["Asha <asha@example.com>"]
> [tool.poetry.dependencies]
> python = "^3.12"
> requests = "^2.31"
> [tool.poetry.group.dev.dependencies]    # (even older: [tool.poetry.dev-dependencies])
> pytest = "^8.0"
> ```
> Poetry 2 still reads it, but new projects should use `[project]`.

## 3. Version constraints

| Constraint | Meaning | Equivalent |
|---|---|---|
| `^2.31.0` (caret) | allow non-breaking updates (SemVer) | `>=2.31.0,<3.0.0` |
| `^0.28.1` | for 0.x, the minor is "major" | `>=0.28.1,<0.29.0` |
| `~2.31.0` (tilde) | patch updates only | `>=2.31.0,<2.32.0` |
| `~2.31` | | `>=2.31,<2.32` |
| `2.*` | wildcard | `>=2.0,<3.0` |
| `>=2.31,<3` | explicit range | |
| `==2.31.0` | exact | |

`poetry add requests` picks the latest version and adds a **caret-style** range, written in PEP 508 form: `requests (>=2.34.2,<3.0.0)`.

> 💡 Upper bounds (`<3.0.0`) protect *applications* from surprise major upgrades, but in *libraries* they can cause dependency conflicts for your users. Many library authors prefer only a lower bound (`poetry add "requests>=2.31"`).

## 4. Adding and removing dependencies

In [4]:
%%bash
cd "$LAB/bookshelf"
poetry add requests 2>&1 | tail -3                    # main dependency (creates .venv and poetry.lock)
poetry add "rich@^13" 2>&1 | tail -1                  # with a caret constraint (@ syntax)
poetry add --group dev pytest 2>&1 | tail -1          # dev group  (shortcut: -D / -G dev)
poetry add -G lint ruff 2>&1 | tail -1                # any named group
poetry add --optional yaml pyyaml 2>&1 | tail -1      # an extra: pip install "bookshelf[yaml]"
cat pyproject.toml

  - Installing requests (2.34.2)

Writing lock file


Writing lock file


Writing lock file


Writing lock file


Writing lock file


...

mail = "asha@example.com"}
]
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "r

equests (>=2.34.2,<3.0.0)",
    "rich (>=13,<14)"
]

[project.optional-dependencies]
yaml = ["pyyaml

 (>=6.0.3,<7.0.0)"]
[tool.poetry]
packages = [{include = "bookshelf", from = "src"}]

[build-system]


requires = ["poetry-core>=2.0.0,<3.0.0"]
build-backend = "poetry.core.masonry.api"

[dependency-gro

ups]
dev = [
    "pytest (>=9.1.1,<10.0.0)"
]
lint = [
    "ruff (>=0.17.1,<0.18.0)"
]


Notice Poetry 2.5 writes dev groups to the standard PEP 735 **`[dependency-groups]`** table, and extras to `[project.optional-dependencies]`.

### More ways to add

```bash
poetry add "httpx[http2]"                                     # extras of a dependency
poetry add "pywin32; sys_platform == 'win32'"                 # or --markers / --platform / --python
poetry add git+https://github.com/encode/httpx.git#0.28.1     # Git (tag/branch/commit after #)
poetry add ../mylib --editable                                # local path (editable)
poetry add https://example.com/pkg-1.0-py3-none-any.whl       # URL
poetry add --allow-prereleases somepkg
poetry add --dry-run httpx                                    # show what would happen
poetry add --lock httpx                                       # only update poetry.lock, don't install
poetry add httpx --source company                             # from a configured private source
```

In [5]:
%%bash
cd "$LAB/bookshelf"
poetry remove rich 2>&1 | tail -1
poetry remove --group lint ruff 2>&1 | tail -1
grep -A3 "dependencies = \[" pyproject.toml | head -5

Writing lock file


Writing lock file


dependencies = [
    "requests (>=2.34.2,<3.0.0)",
]



## 5. The lock file and installing

`poetry.lock` pins the **exact** version and hashes of every package. ✅ Commit it.

| Command | What it does |
|---|---|
| `poetry lock` | re-lock after editing `pyproject.toml` by hand (keeps current pins where possible) |
| `poetry lock --regenerate` | re-solve from scratch |
| `poetry install` | create the venv and install everything from the lock **+ the project itself** (editable) |
| `poetry install --without dev` / `--with docs` / `--only main` | choose groups |
| `poetry install --no-root` | only dependencies (not your project), useful in Docker |
| `poetry install -E yaml` / `--all-extras` / `--all-groups` | extras/groups |
| `poetry sync` | like `install`, but also **removes** packages not in the lock (⚠️ replaces `install --sync`) |
| `poetry update` | upgrade dependencies within their constraints and re-lock |
| `poetry update requests` | upgrade just one |
| `poetry check --lock` | verify `pyproject.toml` is valid and the lock is in sync (CI) |

> ⚠️ **Changed in Poetry 2:** `poetry lock --no-update` was removed, because "don't update" is now the default behaviour of `poetry lock`.

In [6]:
%%bash
cd "$LAB/bookshelf"
head -16 poetry.lock
echo "..."
poetry check --lock
poetry install 2>&1 | tail -3
poetry sync --without dev --dry-run 2>&1 | tail -4

# This file is automatically @generated by Poetry 2.5.1 and should not be changed by hand.

[[packag

e]]
name = "certifi"
version = "2026.10.19"
description = "Python package for providing Mozilla's CA

 Bundle."
optional = false
python-versions = ">=3.7"
groups = ["main"]
files = [
    {file = "certif

i-2026.10.19-py3-none-any.whl", hash = "sha256:42d35340cc5b9ba78e818ddbff0c5b1c2b779de448a340fd1b989

f5fa8670e1b"},
    {file = "certifi-2026.10.19.tar.gz", hash = "sha256:035c1e85572b141bf1c3a3eff0e33

...

...


All set!


No dependencies to install or update

Installing the current project: bookshelf (0.1.0)


  - Installing urllib3 (2.8.0): Skipped for the following reason: Already installed

Installing the 

current project: bookshelf (0.1.0)



## 6. Virtual environments

By default Poetry stores venvs in a central cache folder (`virtualenvs.path`). Many teams set **`virtualenvs.in-project = true`** to get a `.venv` in the project (what this sandbox does), which editors like VS Code detect automatically.

In [7]:
%%bash
cd "$LAB/bookshelf"
poetry env info 2>&1 | head -8
poetry env list
poetry env activate                  # PRINTS the command to activate the venv


Virtualenv
Python:         3.15.0
Implementation: CPython
Path:           /private/var/folders/96/f

6xk04z91_gdystl3qfqv33r0000gn/T/poetry-lab-6ambx1ny/bookshelf/.venv
Executable:     /private/var/fol

ders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/poetry-lab-6ambx1ny/bookshelf/.venv/bin/python
Valid:      

    True



.venv (Activated)


. /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/poetry-lab-6ambx1ny/bookshelf/.venv/bin/a

ctivate


| Command | Purpose |
|---|---|
| `poetry env use python3.13` / `poetry env use /path/to/python` | create/switch the venv's Python |
| `poetry env info [--path]` | details |
| `poetry env list` / `poetry env remove --all` | list / delete venvs |
| `eval $(poetry env activate)` | activate in your current shell (bash/zsh) |
| `poetry python install 3.15` | 🧪 Poetry can download Python too (experimental) |

> ⚠️ **`poetry shell` was removed from core in Poetry 2.** Use `eval $(poetry env activate)` (or `poetry run`), or install the `poetry-plugin-shell` plugin.

### Running commands in the environment

In [8]:
%%bash
cd "$LAB/bookshelf"
cat > src/bookshelf/__init__.py <<'EOF'
def count_words(text: str) -> int:
    return len(text.split())

def main() -> None:
    print("Bookshelf: words in title =", count_words("Fluent Python, 2nd Edition"))
EOF
cat > tests/test_words.py <<'EOF'
from bookshelf import count_words
def test_count_words():
    assert count_words("a b c") == 3
EOF
poetry run pytest -q -p no:cacheprovider 2>&1 | tail -1
poetry run python -c "import requests, bookshelf; print(requests.__version__)"

1 passed in 0.00s


2.34.2


## 7. Scripts (console commands)

In [9]:
%%bash
cd "$LAB/bookshelf"
cat >> pyproject.toml <<'EOF'

[project.scripts]
bookshelf = "bookshelf:main"
EOF
poetry install 2>&1 | tail -1          # re-install the project to create the command
poetry run bookshelf

Installing the current project: bookshelf (0.1.0)


Bookshelf: words in title = 4


## 8. Inspecting dependencies

In [10]:
%%bash
cd "$LAB/bookshelf"
poetry show --top-level
echo "--- tree:"
poetry show --tree requests
echo "--- why is idna installed?"
poetry show --why --tree idna 2>&1 | head -6
echo "--- outdated:"
poetry show --outdated 2>&1 | head -5 || true

pytest             9.1.1      pytest: simple powerful testing with Python
pyyaml         (!) 6.0.3  

    YAML parser and emitter for Python


requests           2.34.2     Python HTTP for Humans.


--- tree:


requests 2.34.2 Python HTTP for Humans.


├── certifi >=2023.5.7
├── charset-normalizer >=2,<4


├── idna >=2.5,<4
└── urllib3 >=1.26,<3


--- why is idna installed?


requests 2.34.2 Python HTTP for Humans.
└── idna >=2.5,<4


--- outdated:


In `poetry show`, **`(!)`** marks a package that is locked but **not installed** (here the optional `yaml` extra). `poetry search <name>` searches configured sources; `poetry debug info` shows environment details; `poetry debug resolve httpx` explains resolution.

## 9. Versioning

In [11]:
%%bash
cd "$LAB/bookshelf"
poetry version
poetry version minor
poetry version prerelease
poetry version --next-phase prerelease
poetry version --short

bookshelf 0.1.0


Bumping version from 0.1.0 to 0.2.0


Bumping version from 0.2.0 to 0.2.1a0


Bumping version from 0.2.1a0 to 0.2.1b0


0.2.1b0


Bump rules: `patch`, `minor`, `major`, `prepatch`, `preminor`, `premajor`, `prerelease` (+ `--next-phase` to go alpha → beta → rc), or an explicit version `poetry version 1.0.0`.

## 10. Building and publishing

In [12]:
%%bash
cd "$LAB/bookshelf"
poetry version 1.0.0 >/dev/null
poetry build 2>&1 | tail -3
ls dist

  - Building wheel
  - Built bookshelf-1.0.0-py3-none-any.whl


bookshelf-1.0.0-py3-none-any.whl
bookshelf-1.0.0.tar.gz


```bash
poetry build --format wheel           # only a wheel (or sdist); --clean, -o OUTPUT
poetry config pypi-token.pypi pypi-AgEI...          # store a PyPI API token (once)
poetry publish                        # upload dist/* to PyPI
poetry publish --build                # build + upload in one go
poetry publish --dry-run              # check everything except the upload
poetry publish -r testpypi            # another repository (configure it first, below)
```
Full publishing walk-through (TestPyPI, tokens, trusted publishing) in lesson 07.

## 11. Private package sources & credentials

In [13]:
%%bash
cd "$LAB/bookshelf"
poetry source add --priority=supplemental company https://pypi.company.example/simple/ 2>&1 | tail -1
poetry source show
sed -n '/tool.poetry.source/,$p' pyproject.toml

Adding source with name company.


 name      : company                              
 url       : https://pypi.company.example/simple/

 
 priority  : supplemental                         

PyPI is implicitly enabled as a primary source

. If you wish to disable it, or alter its priority please refer to https://python-poetry.org/docs/re

positories/#package-sources.



[[tool.poetry.source]]
name = "company"
url = "https://pypi.company.example/simple/"
priority = "sup

plemental"

[build-system]
requires = ["poetry-core>=2.0.0,<3.0.0"]
build-backend = "poetry.core.mas

onry.api"

[dependency-groups]
dev = [
    "pytest (>=9.1.1,<10.0.0)"
]

[project.scripts]
bookshelf

 = "bookshelf:main"


| Priority | Meaning |
|---|---|
| `primary` | searched first (replaces PyPI when it's the only primary) |
| `supplemental` | searched only if not found in primary sources |
| `explicit` | used **only** for packages that name it: `poetry add pkg --source company` |

Credentials (never commit them!):
```bash
poetry config http-basic.company USERNAME PASSWORD      # stored in your keyring if available
export POETRY_HTTP_BASIC_COMPANY_USERNAME=me             # or via environment variables (CI)
export POETRY_HTTP_BASIC_COMPANY_PASSWORD=secret
poetry config repositories.testpypi https://test.pypi.org/legacy/   # for publishing
poetry config pypi-token.testpypi pypi-AgEI...
```

## 12. Configuration

In [14]:
%%bash
cd "$LAB/bookshelf"
poetry config virtualenvs.in-project
poetry config --local installer.max-workers 4           # --local writes poetry.toml in the project
cat poetry.toml
poetry config --list 2>&1 | grep -v -- '-dir' | head -12

true


[installer]
max-workers = 4


installer.builtin-uninstall = false
installer.max-workers = 4
installer.no-binary = null
installer.o

nly-binary = null
installer.parallel = true
installer.re-resolve = false
keyring.enabled = true
repo

sitories.company.url = "https://pypi.company.example/simple/"
requests.max-retries = 0
solver.lazy-w

heel = true
solver.min-release-age = 0
solver.min-release-age-exclude = null


Common settings: `virtualenvs.in-project`, `virtualenvs.create` (set `false` inside Docker), `virtualenvs.use-poetry-python` (⚠️ replaced 1.x `prefer-active-python`), `installer.parallel`, `installer.max-workers`, `solver.min-release-age` (ignore very new releases, a supply-chain safety feature), `keyring.enabled`. Every setting can be an environment variable: `POETRY_VIRTUALENVS_IN_PROJECT=true`.

## 13. Plugins & `poetry self`

Poetry's own environment can be extended with plugins:
```bash
poetry self add poetry-plugin-export        # adds `poetry export -f requirements.txt -o requirements.txt`
poetry self add poetry-plugin-shell         # brings back `poetry shell`
poetry self add poetry-dynamic-versioning   # version from Git tags
poetry self show plugins
poetry self update                          # upgrade Poetry itself
```
> ⚠️ **Changed in Poetry 2:** `poetry export` is no longer bundled; install `poetry-plugin-export`.

## 14. Poetry in Docker and CI

```dockerfile
FROM python:3.15-slim
ENV POETRY_VERSION=2.5.1 POETRY_VIRTUALENVS_CREATE=false POETRY_NO_INTERACTION=1
RUN pip install "poetry==$POETRY_VERSION"
WORKDIR /app
COPY pyproject.toml poetry.lock ./
RUN poetry install --only main --no-root          # dependencies (cached layer)
COPY . .
RUN poetry install --only main                    # the project itself
CMD ["bookshelf"]
```
```yaml
# GitHub Actions
- run: pipx install poetry
- uses: actions/setup-python@v6
  with: { python-version: "3.15", cache: "poetry" }
- run: poetry install --with dev
- run: poetry check --lock && poetry run pytest
```

## 15. Poetry vs uv

| | Poetry 2.5 | uv 0.13 |
|---|---|---|
| Written in | Python | Rust (10–100× faster installs/resolution) |
| `pyproject.toml` | `[project]` + `[tool.poetry]` | `[project]` + `[tool.uv]` |
| Lock file | `poetry.lock` | `uv.lock` (universal), export to `pylock.toml` |
| Dev deps | PEP 735 groups | PEP 735 groups |
| Python install | 🧪 experimental | ✅ mature (`uv python install`) |
| Scripts (PEP 723) | ❌ | ✅ |
| Global tools (pipx-like) | ❌ | ✅ `uvx`, `uv tool` |
| Build backend | `poetry-core` | `uv_build` (or any) |
| Publish | ✅ | ✅ |
| Workspaces / monorepos | via plugins | ✅ built in |
| Maturity / ecosystem | very mature, many plugins | newer, very fast-moving |

**Migrating Poetry → uv:** `uvx migrate-to-uv` converts the `pyproject.toml` and lock data; then `uv lock && uv sync`.
**Migrating to Poetry:** `poetry init` in an existing folder, then `poetry add` your requirements (`cat requirements.txt | xargs poetry add`).

## 📝 Summary
- Install Poetry with pipx/uv tool; create projects with `poetry new`; Poetry 2 uses the standard `[project]` table.
- `poetry add` (groups `-G`, extras `--optional`, sources), `remove`, `lock`, `install`, `sync`, `update`, `check --lock`.
- Environments: `env info/use/list/remove/activate`, `poetry run`; `poetry shell` and `export` are now plugins.
- `show --tree/--outdated/--why`, `version` bumps, `build`, `publish`, private `source`s with priorities.

## 🏋️ Practice
1. `poetry new --flat notes`, add `rich` and a `dev` group with `pytest`, write a test, run it with `poetry run pytest`.
2. Add a console script `notes = "notes:main"` and run it with `poetry run notes`.
3. Bump the version to `0.2.0b1`, then to the next phase (`rc`), then build the package.
4. Configure an `explicit` source and add one package from it (it doesn't need to exist for `--dry-run`).
5. Compare: create the same project with `uv init` (lesson 04). Which `pyproject.toml` sections are identical?